# A/B Testing: A Comprehensive Guide

## What is A/B Testing?

A/B testing (also known as **split testing** or **randomized controlled experiment**) is a statistical methodology for comparing two or more variants of a treatment to determine which performs better against a defined metric. It is the gold standard for establishing **causal relationships** between changes and outcomes.

### Core Principle

Randomly split users/subjects into groups → expose each group to a different variant → measure outcomes → use statistical inference to determine if differences are real or due to chance.

### Why A/B Testing Matters

| Aspect | Without A/B Testing | With A/B Testing |
| --- | --- | --- |
| Decision Basis | Opinions, HiPPO (Highest Paid Person's Opinion) | Data-driven evidence |
| Causality | Correlation only | Causal inference |
| Risk | Ship changes blindly | Quantified impact before full rollout |
| Learning | Anecdotal | Systematic, reproducible |

## 1. Experimental Design & Setup

### The A/B Testing Lifecycle

```
1. Define Hypothesis → 2. Choose Metrics → 3. Calculate Sample Size → 4. Randomize & Assign
→ 5. Run Experiment → 6. Analyze Results → 7. Make Decision
```

### Key Components

**1. Hypothesis Formulation**
- **Null Hypothesis** ($$H_0$$): There is no difference between control and treatment
- **Alternative Hypothesis** ($$H_1$$): There is a statistically significant difference

**2. Metric Selection**
- **Primary Metric (OEC - Overall Evaluation Criterion)**: The single metric that determines success (e.g., conversion rate, revenue per user)
- **Guardrail Metrics**: Metrics that must not degrade (e.g., page load time, crash rate)
- **Secondary Metrics**: Supporting signals that help explain results

**3. Randomization Unit**
- User-level (most common)
- Session-level
- Page-view level
- Device-level

**4. Key Parameters**
- **Significance Level** ($$\alpha$$): Probability of Type I error (false positive). Typically 0.05
- **Statistical Power** ($$1 - \beta$$): Probability of detecting a true effect. Typically 0.80
- **Minimum Detectable Effect (MDE)**: The smallest effect worth detecting
- **Baseline Rate**: Current performance of the control

In [0]:
# A/B Testing - Required Libraries
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm, t, chi2_contingency, mannwhitneyu, ttest_ind, proportion
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.power import TTestIndPower, NormalIndPower, zt_ind_solve_power
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print("Libraries loaded successfully!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

In [0]:
# Generate comprehensive synthetic A/B test dataset
np.random.seed(42)

# Simulating an e-commerce A/B test
# Control: Original checkout page | Treatment: Redesigned checkout page
n_control = 5000
n_treatment = 5000

# --- Conversion Rate Data ---
# Control: 12% conversion rate | Treatment: 13.5% conversion rate (12.5% relative lift)
control_conversions = np.random.binomial(1, 0.12, n_control)
treatment_conversions = np.random.binomial(1, 0.135, n_treatment)

# --- Revenue Data (continuous) ---
# Revenue per converting user
control_revenue = np.where(control_conversions == 1, 
                           np.random.lognormal(mean=3.5, sigma=0.8, size=n_control), 0)
treatment_revenue = np.where(treatment_conversions == 1, 
                              np.random.lognormal(mean=3.6, sigma=0.75, size=n_treatment), 0)

# --- Time on Page Data ---
control_time = np.random.exponential(scale=45, size=n_control)  # seconds
treatment_time = np.random.exponential(scale=52, size=n_treatment)  # seconds

# --- Bounce Rate Data ---
control_bounce = np.random.binomial(1, 0.35, n_control)
treatment_bounce = np.random.binomial(1, 0.30, n_treatment)

# Create DataFrame
control_df = pd.DataFrame({
    'user_id': range(1, n_control + 1),
    'group': 'control',
    'converted': control_conversions,
    'revenue': np.round(control_revenue, 2),
    'time_on_page': np.round(control_time, 1),
    'bounced': control_bounce,
    'device': np.random.choice(['mobile', 'desktop', 'tablet'], n_control, p=[0.55, 0.35, 0.10]),
    'day': np.random.randint(1, 15, n_control)
})

treatment_df = pd.DataFrame({
    'user_id': range(n_control + 1, n_control + n_treatment + 1),
    'group': 'treatment',
    'converted': treatment_conversions,
    'revenue': np.round(treatment_revenue, 2),
    'time_on_page': np.round(treatment_time, 1),
    'bounced': treatment_bounce,
    'device': np.random.choice(['mobile', 'desktop', 'tablet'], n_treatment, p=[0.55, 0.35, 0.10]),
    'day': np.random.randint(1, 15, n_treatment)
})

ab_data = pd.concat([control_df, treatment_df], ignore_index=True)

print("=" * 60)
print("SYNTHETIC A/B TEST DATASET")
print("=" * 60)
print(f"\nScenario: E-commerce Checkout Page Redesign")
print(f"Control: Original checkout page")
print(f"Treatment: Redesigned checkout page")
print(f"\nTotal Users: {len(ab_data):,}")
print(f"Control Group: {n_control:,} users")
print(f"Treatment Group: {n_treatment:,} users")
print(f"\n--- Summary Statistics ---")
print(f"\nConversion Rate:")
print(f"  Control:   {control_conversions.mean():.4f} ({control_conversions.mean()*100:.2f}%)")
print(f"  Treatment: {treatment_conversions.mean():.4f} ({treatment_conversions.mean()*100:.2f}%)")
print(f"  Lift:      {((treatment_conversions.mean() - control_conversions.mean()) / control_conversions.mean()) * 100:.2f}%")
print(f"\nAvg Revenue Per User:")
print(f"  Control:   ${control_revenue.mean():.2f}")
print(f"  Treatment: ${treatment_revenue.mean():.2f}")
print(f"\nBounce Rate:")
print(f"  Control:   {control_bounce.mean():.4f} ({control_bounce.mean()*100:.2f}%)")
print(f"  Treatment: {treatment_bounce.mean():.4f} ({treatment_bounce.mean()*100:.2f}%)")

display(ab_data.head(10))

## 2. Sampling & Sample Size Calculation

### Why Sample Size Matters

Running an experiment with too few users leads to **underpowered tests** (can't detect real effects). Running too long wastes opportunity cost and exposes users to potentially inferior experiences.

### Sample Size Formula for Proportions

For a two-proportion z-test:

$$n = \frac{(Z_{\alpha/2} + Z_{\beta})^2 \cdot [p_1(1-p_1) + p_2(1-p_2)]}{(p_1 - p_2)^2}$$

Where:
- $$n$$ = sample size per group
- $$Z_{\alpha/2}$$ = z-score for significance level (1.96 for $$\alpha = 0.05$$)
- $$Z_{\beta}$$ = z-score for power (0.84 for power = 0.80)
- $$p_1$$ = baseline conversion rate
- $$p_2$$ = expected conversion rate with treatment

### Sample Size Formula for Continuous Metrics

$$n = \frac{2(Z_{\alpha/2} + Z_{\beta})^2 \cdot \sigma^2}{\delta^2}$$

Where:
- $$\sigma$$ = pooled standard deviation
- $$\delta$$ = minimum detectable effect (absolute difference in means)

### Practical Considerations
- **Traffic volume**: How long to reach required sample size?
- **Novelty/Primacy effects**: New designs may initially attract more attention
- **Day-of-week effects**: Run for full weeks to avoid bias
- **Multiple comparisons**: Adjust sample size if testing multiple variants

In [0]:
# ============================================================
# SAMPLE SIZE CALCULATION
# ============================================================

def calculate_sample_size_proportion(baseline_rate, mde_relative, alpha=0.05, power=0.80):
    """
    Calculate required sample size for a two-proportion z-test.
    
    Parameters:
    -----------
    baseline_rate : float - Current conversion rate (e.g., 0.12)
    mde_relative : float - Minimum detectable effect as relative change (e.g., 0.10 for 10% lift)
    alpha : float - Significance level
    power : float - Statistical power
    
    Returns:
    --------
    int : Required sample size per group
    """
    p1 = baseline_rate
    p2 = baseline_rate * (1 + mde_relative)
    
    # Effect size (Cohen's h)
    effect_size = 2 * np.arcsin(np.sqrt(p2)) - 2 * np.arcsin(np.sqrt(p1))
    
    # Using statsmodels
    analysis = NormalIndPower()
    sample_size = analysis.solve_power(
        effect_size=effect_size,
        alpha=alpha,
        power=power,
        alternative='two-sided'
    )
    
    return int(np.ceil(sample_size))


def calculate_sample_size_continuous(baseline_mean, baseline_std, mde_absolute, alpha=0.05, power=0.80):
    """
    Calculate required sample size for a two-sample t-test.
    
    Parameters:
    -----------
    baseline_mean : float - Current metric mean
    baseline_std : float - Current metric standard deviation
    mde_absolute : float - Minimum detectable effect (absolute)
    alpha : float - Significance level
    power : float - Statistical power
    
    Returns:
    --------
    int : Required sample size per group
    """
    # Cohen's d
    effect_size = mde_absolute / baseline_std
    
    analysis = TTestIndPower()
    sample_size = analysis.solve_power(
        effect_size=effect_size,
        alpha=alpha,
        power=power,
        alternative='two-sided'
    )
    
    return int(np.ceil(sample_size))


# --- Example Calculations ---
print("=" * 60)
print("SAMPLE SIZE CALCULATIONS")
print("=" * 60)

# Scenario 1: Conversion rate test
baseline_cvr = 0.12
relative_mde = 0.10  # Want to detect at least 10% relative lift

n_prop = calculate_sample_size_proportion(baseline_cvr, relative_mde)
print(f"\n--- Scenario 1: Conversion Rate Test ---")
print(f"Baseline CVR: {baseline_cvr*100}%")
print(f"MDE: {relative_mde*100}% relative lift (absolute: {baseline_cvr*relative_mde*100:.2f}pp)")
print(f"Required sample per group: {n_prop:,}")
print(f"Total sample needed: {2*n_prop:,}")
print(f"At 1000 users/day: ~{np.ceil(2*n_prop/1000):.0f} days")

# Scenario 2: Revenue per user test
baseline_rev_mean = 4.5
baseline_rev_std = 15.0  # High variance typical for revenue
mde_rev = 0.50  # Want to detect $0.50 increase

n_cont = calculate_sample_size_continuous(baseline_rev_mean, baseline_rev_std, mde_rev)
print(f"\n--- Scenario 2: Revenue Per User Test ---")
print(f"Baseline Revenue/User: ${baseline_rev_mean:.2f} (std: ${baseline_rev_std:.2f})")
print(f"MDE: ${mde_rev:.2f} absolute increase")
print(f"Required sample per group: {n_cont:,}")
print(f"Total sample needed: {2*n_cont:,}")
print(f"At 1000 users/day: ~{np.ceil(2*n_cont/1000):.0f} days")

# --- Power Curve Visualization ---
print(f"\n--- Power Curve ---")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Sample size vs MDE for proportions
mde_range = np.arange(0.05, 0.31, 0.01)
sample_sizes = [calculate_sample_size_proportion(0.12, mde) for mde in mde_range]

axes[0].plot(mde_range * 100, sample_sizes, 'b-', linewidth=2)
axes[0].axhline(y=5000, color='r', linestyle='--', label='n=5000 (our experiment)')
axes[0].set_xlabel('Minimum Detectable Effect (% relative lift)', fontsize=11)
axes[0].set_ylabel('Required Sample Size Per Group', fontsize=11)
axes[0].set_title('Sample Size vs MDE\n(Baseline CVR = 12%, α=0.05, Power=0.80)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Power vs Sample Size
n_range = np.arange(500, 20001, 500)
effect_size_h = 2 * np.arcsin(np.sqrt(0.135)) - 2 * np.arcsin(np.sqrt(0.12))
analysis = NormalIndPower()
powers = [analysis.power(effect_size=effect_size_h, nobs1=n, alpha=0.05) for n in n_range]

axes[1].plot(n_range, powers, 'g-', linewidth=2)
axes[1].axhline(y=0.80, color='r', linestyle='--', label='80% Power Threshold')
axes[1].axvline(x=5000, color='orange', linestyle='--', label='n=5000 (our experiment)')
axes[1].set_xlabel('Sample Size Per Group', fontsize=11)
axes[1].set_ylabel('Statistical Power', fontsize=11)
axes[1].set_title('Power vs Sample Size\n(Effect: 12% → 13.5% CVR, α=0.05)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 3. P-Values & Hypothesis Testing

### What is a P-Value?

The p-value is the probability of observing a result **at least as extreme** as the one obtained, assuming the null hypothesis is true.

$$p\text{-value} = P(\text{observing data} \mid H_0 \text{ is true})$$

### Decision Framework

| P-value | Interpretation | Action |
| --- | --- | --- |
| p < 0.01 | Strong evidence against $$H_0$$ | Reject $$H_0$$ with high confidence |
| 0.01 ≤ p < 0.05 | Moderate evidence against $$H_0$$ | Reject $$H_0$$ (standard threshold) |
| 0.05 ≤ p < 0.10 | Weak evidence | Borderline - consider context |
| p ≥ 0.10 | Insufficient evidence | Fail to reject $$H_0$$ |

### Types of Errors

| | $$H_0$$ is True (No Effect) | $$H_0$$ is False (Real Effect) |
| --- | --- | --- |
| **Reject $$H_0$$** | Type I Error ($$\alpha$$) - False Positive | Correct Decision (Power = $$1-\beta$$) |
| **Fail to Reject $$H_0$$** | Correct Decision | Type II Error ($$\beta$$) - False Negative |

### Common Misconceptions
1. **P-value ≠ probability that $$H_0$$ is true** — it's the probability of the DATA given $$H_0$$
2. **Statistical significance ≠ practical significance** — a tiny effect can be "significant" with enough data
3. **p > 0.05 ≠ "no effect"** — it means insufficient evidence to conclude there IS an effect
4. **P-values don't measure effect size** — always report confidence intervals alongside

In [0]:
# ============================================================
# P-VALUE DEMONSTRATION & INTERPRETATION
# ============================================================

# Demonstrate what p-values look like under H0 (no effect) vs H1 (real effect)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simulation: 10000 A/B tests where there IS NO real difference (H0 is true)
np.random.seed(123)
n_simulations = 10000
p_values_null = []

for _ in range(n_simulations):
    # Both groups have same conversion rate (H0 true)
    group_a = np.random.binomial(1, 0.12, 1000)
    group_b = np.random.binomial(1, 0.12, 1000)
    
    # Two-proportion z-test
    count = np.array([group_a.sum(), group_b.sum()])
    nobs = np.array([len(group_a), len(group_b)])
    _, p_val = proportions_ztest(count, nobs)
    p_values_null.append(p_val)

# Simulation: 10000 A/B tests where there IS a real difference (H1 true)
p_values_alt = []
for _ in range(n_simulations):
    group_a = np.random.binomial(1, 0.12, 1000)
    group_b = np.random.binomial(1, 0.15, 1000)  # True 25% relative lift
    
    count = np.array([group_a.sum(), group_b.sum()])
    nobs = np.array([len(group_a), len(group_b)])
    _, p_val = proportions_ztest(count, nobs)
    p_values_alt.append(p_val)

# Plot distributions
axes[0].hist(p_values_null, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
axes[0].set_title('P-value Distribution Under H₀ (No Effect)\nExpect: Uniform Distribution', fontsize=12)
axes[0].set_xlabel('P-value', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].legend(fontsize=10)
false_positive_rate = np.mean(np.array(p_values_null) < 0.05)
axes[0].text(0.5, 0.85, f'False Positive Rate: {false_positive_rate:.3f}\n(Expected: ~0.05)', 
            transform=axes[0].transAxes, fontsize=11, 
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

axes[1].hist(p_values_alt, bins=50, edgecolor='black', alpha=0.7, color='forestgreen')
axes[1].axvline(x=0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
axes[1].set_title('P-value Distribution Under H₁ (Real Effect: 25% lift)\nExpect: Skewed Left', fontsize=12)
axes[1].set_xlabel('P-value', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].legend(fontsize=10)
true_positive_rate = np.mean(np.array(p_values_alt) < 0.05)
axes[1].text(0.5, 0.85, f'Power (True Positive Rate): {true_positive_rate:.3f}\n(n=1000 per group)', 
            transform=axes[1].transAxes, fontsize=11,
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print(f"  Under H₀, ~{false_positive_rate*100:.1f}% of tests show p < 0.05 (Type I errors)")
print(f"  Under H₁, ~{true_positive_rate*100:.1f}% of tests correctly reject H₀ (Power)")

## 4. Statistical Tests for A/B Testing

### Choosing the Right Test

| Scenario | Metric Type | Recommended Test | Assumptions |
| --- | --- | --- | --- |
| Conversion rate (A vs B) | Binary/Proportion | **Z-test for proportions** | Large sample (n > 30) |
| Revenue, time-on-site | Continuous, Normal | **Two-sample t-test** | Normality (or large n via CLT) |
| Revenue, engagement | Continuous, Non-normal | **Mann-Whitney U test** | Independent samples |
| Click distribution across categories | Categorical | **Chi-square test** | Expected freq ≥ 5 |
| Multiple variants (A/B/C/D) | Continuous | **ANOVA + post-hoc** | Normality, equal variance |
| Paired before/after | Continuous | **Paired t-test** | Normality of differences |
| Small sample, non-normal | Any | **Permutation / Bootstrap test** | None (non-parametric) |
| Ratio metrics (CTR per session) | Ratio | **Delta method / Bootstrap** | Complex variance structure |

In [0]:
# ============================================================
# TEST 1: TWO-PROPORTION Z-TEST (Conversion Rates)
# ============================================================
# Use case: Comparing conversion rates between control and treatment

def two_proportion_ztest(data, group_col='group', outcome_col='converted', 
                         control_label='control', treatment_label='treatment'):
    """
    Perform a two-proportion z-test with full reporting.
    """
    control = data[data[group_col] == control_label][outcome_col]
    treatment = data[data[group_col] == treatment_label][outcome_col]
    
    n_c, n_t = len(control), len(treatment)
    p_c, p_t = control.mean(), treatment.mean()
    
    # Pooled proportion
    p_pool = (control.sum() + treatment.sum()) / (n_c + n_t)
    
    # Standard error
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))
    
    # Z-statistic
    z_stat = (p_t - p_c) / se
    
    # P-value (two-tailed)
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))
    
    # Confidence interval for the difference
    se_diff = np.sqrt(p_c*(1-p_c)/n_c + p_t*(1-p_t)/n_t)
    ci_lower = (p_t - p_c) - 1.96 * se_diff
    ci_upper = (p_t - p_c) + 1.96 * se_diff
    
    # Relative lift
    relative_lift = (p_t - p_c) / p_c
    
    return {
        'test': 'Two-Proportion Z-Test',
        'control_rate': p_c,
        'treatment_rate': p_t,
        'absolute_diff': p_t - p_c,
        'relative_lift': relative_lift,
        'z_statistic': z_stat,
        'p_value': p_value,
        'ci_95': (ci_lower, ci_upper),
        'significant': p_value < 0.05,
        'n_control': n_c,
        'n_treatment': n_t
    }

# Run the test
result_ztest = two_proportion_ztest(ab_data)

print("=" * 60)
print("TEST 1: TWO-PROPORTION Z-TEST")
print("Metric: Conversion Rate")
print("=" * 60)
print(f"\n  Control Conversion Rate:   {result_ztest['control_rate']:.4f} ({result_ztest['control_rate']*100:.2f}%)")
print(f"  Treatment Conversion Rate: {result_ztest['treatment_rate']:.4f} ({result_ztest['treatment_rate']*100:.2f}%)")
print(f"  Absolute Difference:       {result_ztest['absolute_diff']:.4f} ({result_ztest['absolute_diff']*100:.2f}pp)")
print(f"  Relative Lift:             {result_ztest['relative_lift']*100:.2f}%")
print(f"\n  Z-Statistic: {result_ztest['z_statistic']:.4f}")
print(f"  P-Value:     {result_ztest['p_value']:.6f}")
print(f"  95% CI:      [{result_ztest['ci_95'][0]*100:.2f}pp, {result_ztest['ci_95'][1]*100:.2f}pp]")
print(f"\n  Decision: {'REJECT H₀ - Statistically Significant! ✅' if result_ztest['significant'] else 'FAIL TO REJECT H₀ - Not Significant ❌'}")

# Verify with statsmodels
count = np.array([ab_data[ab_data['group']=='treatment']['converted'].sum(),
                  ab_data[ab_data['group']=='control']['converted'].sum()])
nobs = np.array([len(ab_data[ab_data['group']=='treatment']),
                 len(ab_data[ab_data['group']=='control'])])
z_sm, p_sm = proportions_ztest(count, nobs)
print(f"\n  [Verification via statsmodels: z={z_sm:.4f}, p={p_sm:.6f}]")

In [0]:
# ============================================================
# TEST 2: TWO-SAMPLE T-TEST (Revenue Per User)
# ============================================================
# Use case: Comparing mean revenue between groups (continuous metric)

def two_sample_ttest(data, group_col='group', metric_col='revenue',
                    control_label='control', treatment_label='treatment',
                    equal_var=False):
    """
    Perform Welch's t-test (unequal variance by default) with full reporting.
    """
    control = data[data[group_col] == control_label][metric_col]
    treatment = data[data[group_col] == treatment_label][metric_col]
    
    n_c, n_t = len(control), len(treatment)
    mean_c, mean_t = control.mean(), treatment.mean()
    std_c, std_t = control.std(), treatment.std()
    
    # Welch's t-test
    t_stat, p_value = ttest_ind(treatment, control, equal_var=equal_var)
    
    # Confidence interval (Welch-Satterthwaite)
    se_diff = np.sqrt(std_c**2/n_c + std_t**2/n_t)
    df = ((std_c**2/n_c + std_t**2/n_t)**2) / \
         ((std_c**2/n_c)**2/(n_c-1) + (std_t**2/n_t)**2/(n_t-1))
    t_crit = t.ppf(0.975, df)
    ci_lower = (mean_t - mean_c) - t_crit * se_diff
    ci_upper = (mean_t - mean_c) + t_crit * se_diff
    
    return {
        'test': "Welch's Two-Sample T-Test",
        'control_mean': mean_c,
        'treatment_mean': mean_t,
        'control_std': std_c,
        'treatment_std': std_t,
        'absolute_diff': mean_t - mean_c,
        'relative_lift': (mean_t - mean_c) / mean_c if mean_c != 0 else np.nan,
        't_statistic': t_stat,
        'p_value': p_value,
        'degrees_of_freedom': df,
        'ci_95': (ci_lower, ci_upper),
        'significant': p_value < 0.05
    }

# Run the test on revenue
result_ttest = two_sample_ttest(ab_data, metric_col='revenue')

print("=" * 60)
print("TEST 2: WELCH'S TWO-SAMPLE T-TEST")
print("Metric: Revenue Per User")
print("=" * 60)
print(f"\n  Control Mean Revenue:   ${result_ttest['control_mean']:.4f} (std: ${result_ttest['control_std']:.2f})")
print(f"  Treatment Mean Revenue: ${result_ttest['treatment_mean']:.4f} (std: ${result_ttest['treatment_std']:.2f})")
print(f"  Absolute Difference:    ${result_ttest['absolute_diff']:.4f}")
print(f"  Relative Lift:          {result_ttest['relative_lift']*100:.2f}%")
print(f"\n  T-Statistic: {result_ttest['t_statistic']:.4f}")
print(f"  P-Value:     {result_ttest['p_value']:.6f}")
print(f"  Degrees of Freedom: {result_ttest['degrees_of_freedom']:.1f}")
print(f"  95% CI:      [${result_ttest['ci_95'][0]:.4f}, ${result_ttest['ci_95'][1]:.4f}]")
print(f"\n  Decision: {'REJECT H₀ - Statistically Significant! ✅' if result_ttest['significant'] else 'FAIL TO REJECT H₀ - Not Significant ❌'}")

# Also test time on page
result_time = two_sample_ttest(ab_data, metric_col='time_on_page')
print(f"\n{'='*60}")
print(f"BONUS: Time on Page")
print(f"{'='*60}")
print(f"  Control: {result_time['control_mean']:.1f}s | Treatment: {result_time['treatment_mean']:.1f}s")
print(f"  Lift: {result_time['relative_lift']*100:.1f}% | p-value: {result_time['p_value']:.6f}")
print(f"  Decision: {'Significant ✅' if result_time['significant'] else 'Not Significant ❌'}")

In [0]:
# ============================================================
# TEST 3: CHI-SQUARE TEST OF INDEPENDENCE
# ============================================================
# Use case: Testing if the distribution of a categorical outcome
# differs between groups (e.g., engagement level distribution)

# Create engagement categories
def categorize_engagement(time):
    if time < 15:
        return 'Low'
    elif time < 60:
        return 'Medium'
    elif time < 120:
        return 'High'
    else:
        return 'Very High'

ab_data['engagement'] = ab_data['time_on_page'].apply(categorize_engagement)

# Create contingency table
contingency = pd.crosstab(ab_data['group'], ab_data['engagement'])
print("=" * 60)
print("TEST 3: CHI-SQUARE TEST OF INDEPENDENCE")
print("Question: Does engagement distribution differ between groups?")
print("=" * 60)

print("\n--- Contingency Table ---")
display(contingency)

# Proportions for better comparison
contingency_pct = pd.crosstab(ab_data['group'], ab_data['engagement'], normalize='index') * 100
print("\n--- Proportions (%) ---")
display(contingency_pct.round(2))

# Perform Chi-Square test
chi2, p_value, dof, expected_freq = chi2_contingency(contingency)

# Cramér's V (effect size)
n = contingency.sum().sum()
k = min(contingency.shape) - 1
cramers_v = np.sqrt(chi2 / (n * k))

print(f"\n--- Test Results ---")
print(f"  Chi-Square Statistic: {chi2:.4f}")
print(f"  Degrees of Freedom:   {dof}")
print(f"  P-Value:              {p_value:.6f}")
print(f"  Cramér's V (effect):  {cramers_v:.4f}")
print(f"\n  Decision: {'REJECT H₀ - Distributions differ significantly! ✅' if p_value < 0.05 else 'FAIL TO REJECT H₀ - No significant difference ❌'}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
contingency_pct.plot(kind='bar', ax=ax, width=0.7)
ax.set_xlabel('Group', fontsize=11)
ax.set_ylabel('Percentage (%)', fontsize=11)
ax.set_title(f'Engagement Distribution by Group\n(χ² = {chi2:.2f}, p = {p_value:.4f})', fontsize=12)
ax.legend(title='Engagement Level', fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# TEST 4: MANN-WHITNEY U TEST (Non-Parametric)
# ============================================================
# Use case: When data is heavily skewed (e.g., revenue, session duration)
# Does NOT assume normality - tests if one group tends to have larger values

control_rev = ab_data[ab_data['group'] == 'control']['revenue']
treatment_rev = ab_data[ab_data['group'] == 'treatment']['revenue']

# Check normality first (to justify using Mann-Whitney)
from scipy.stats import shapiro, kstest

# Shapiro-Wilk on a sample (limited to 5000 observations)
sample_size = min(5000, len(control_rev))
_, p_shapiro_c = shapiro(control_rev.sample(sample_size, random_state=42))
_, p_shapiro_t = shapiro(treatment_rev.sample(sample_size, random_state=42))

print("=" * 60)
print("TEST 4: MANN-WHITNEY U TEST (Non-Parametric)")
print("Metric: Revenue Per User (heavily right-skewed)")
print("=" * 60)

print(f"\n--- Normality Check (Shapiro-Wilk) ---")
print(f"  Control p-value:   {p_shapiro_c:.6f} {'(NOT Normal)' if p_shapiro_c < 0.05 else '(Normal)'}")
print(f"  Treatment p-value: {p_shapiro_t:.6f} {'(NOT Normal)' if p_shapiro_t < 0.05 else '(Normal)'}")
print(f"  Conclusion: Data is NOT normally distributed → Mann-Whitney is appropriate")

# Perform Mann-Whitney U test
u_stat, p_value_mw = mannwhitneyu(treatment_rev, control_rev, alternative='two-sided')

# Effect size: rank-biserial correlation
n1, n2 = len(treatment_rev), len(control_rev)
rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

# Median comparison
median_c = control_rev.median()
median_t = treatment_rev.median()

print(f"\n--- Mann-Whitney U Test Results ---")
print(f"  Control Median Revenue:   ${median_c:.2f}")
print(f"  Treatment Median Revenue: ${median_t:.2f}")
print(f"  U-Statistic:              {u_stat:,.0f}")
print(f"  P-Value:                  {p_value_mw:.6f}")
print(f"  Rank-Biserial Correlation: {rank_biserial:.4f}")
print(f"\n  Decision: {'REJECT H₀ - Treatment group tends to have larger values! ✅' if p_value_mw < 0.05 else 'FAIL TO REJECT H₀ ❌'}")

# Visualize the distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue distribution (including zeros)
axes[0].hist(control_rev[control_rev > 0], bins=50, alpha=0.6, label='Control', color='steelblue', density=True)
axes[0].hist(treatment_rev[treatment_rev > 0], bins=50, alpha=0.6, label='Treatment', color='coral', density=True)
axes[0].set_xlabel('Revenue ($)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Revenue Distribution (Converters Only)\nHighly Right-Skewed → Mann-Whitney Preferred', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].set_xlim(0, 200)

# Box plot comparison
ab_data.boxplot(column='revenue', by='group', ax=axes[1])
axes[1].set_title(f'Revenue by Group\n(Mann-Whitney p = {p_value_mw:.4f})', fontsize=11)
axes[1].set_xlabel('Group', fontsize=11)
axes[1].set_ylabel('Revenue ($)', fontsize=11)
axes[1].set_ylim(0, 150)
plt.suptitle('')  # Remove automatic title

plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# TEST 5: BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================
# Use case: When parametric assumptions don't hold, or for complex metrics
# (ratios, percentiles, etc.) where analytical formulas don't exist

def bootstrap_ab_test(control, treatment, n_bootstrap=10000, confidence=0.95, seed=42):
    """
    Perform bootstrap A/B test by resampling the difference in means.
    
    Returns confidence interval for the treatment effect.
    """
    np.random.seed(seed)
    n_c, n_t = len(control), len(treatment)
    
    bootstrap_diffs = []
    for _ in range(n_bootstrap):
        # Resample with replacement
        boot_control = np.random.choice(control, size=n_c, replace=True)
        boot_treatment = np.random.choice(treatment, size=n_t, replace=True)
        
        # Compute difference in means
        bootstrap_diffs.append(boot_treatment.mean() - boot_control.mean())
    
    bootstrap_diffs = np.array(bootstrap_diffs)
    
    # Confidence interval (percentile method)
    alpha = 1 - confidence
    ci_lower = np.percentile(bootstrap_diffs, alpha/2 * 100)
    ci_upper = np.percentile(bootstrap_diffs, (1 - alpha/2) * 100)
    
    # P-value approximation: proportion of bootstrap samples where diff <= 0
    p_value = 2 * min(np.mean(bootstrap_diffs <= 0), np.mean(bootstrap_diffs >= 0))
    
    return {
        'mean_diff': np.mean(bootstrap_diffs),
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'p_value_approx': p_value,
        'bootstrap_distribution': bootstrap_diffs,
        'significant': (ci_lower > 0) or (ci_upper < 0)  # CI doesn't contain 0
    }

# Bootstrap test on revenue
control_rev_arr = ab_data[ab_data['group'] == 'control']['revenue'].values
treatment_rev_arr = ab_data[ab_data['group'] == 'treatment']['revenue'].values

result_bootstrap = bootstrap_ab_test(control_rev_arr, treatment_rev_arr)

print("=" * 60)
print("TEST 5: BOOTSTRAP CONFIDENCE INTERVALS")
print("Metric: Revenue Per User (10,000 bootstrap iterations)")
print("=" * 60)
print(f"\n  Observed Difference:      ${treatment_rev_arr.mean() - control_rev_arr.mean():.4f}")
print(f"  Bootstrap Mean Difference: ${result_bootstrap['mean_diff']:.4f}")
print(f"  95% CI: [${result_bootstrap['ci_lower']:.4f}, ${result_bootstrap['ci_upper']:.4f}]")
print(f"  Approximate P-Value:       {result_bootstrap['p_value_approx']:.4f}")
print(f"\n  Decision: {'CI does not contain 0 → Significant! ✅' if result_bootstrap['significant'] else 'CI contains 0 → Not Significant ❌'}")

# Visualize bootstrap distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(result_bootstrap['bootstrap_distribution'], bins=100, edgecolor='black', 
        alpha=0.7, color='mediumpurple', density=True)
ax.axvline(x=0, color='red', linestyle='-', linewidth=2, label='No Effect (0)')
ax.axvline(x=result_bootstrap['ci_lower'], color='orange', linestyle='--', 
           linewidth=2, label=f'95% CI Lower: ${result_bootstrap["ci_lower"]:.3f}')
ax.axvline(x=result_bootstrap['ci_upper'], color='orange', linestyle='--', 
           linewidth=2, label=f'95% CI Upper: ${result_bootstrap["ci_upper"]:.3f}')
ax.axvline(x=result_bootstrap['mean_diff'], color='green', linestyle='-', 
           linewidth=2, label=f'Mean Diff: ${result_bootstrap["mean_diff"]:.3f}')
ax.set_xlabel('Difference in Revenue (Treatment - Control)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Bootstrap Distribution of Treatment Effect\n(Revenue Per User)', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 5. Advanced A/B Testing Methods

### Sequential Testing (Continuous Monitoring)
Traditional tests require a fixed sample size. Sequential testing allows you to **peek at results** without inflating Type I error.

### Bayesian A/B Testing
Instead of p-values, Bayesian methods give you:
- **Probability of being best**: P(Treatment > Control)
- **Expected loss**: How much you'd lose by choosing the wrong variant
- **Credible intervals**: Direct probability statements about the parameter

### Multi-Armed Bandit (MAB)
Balances **exploration** (learning which variant is best) vs **exploitation** (sending traffic to the best variant). Useful when:
- You want to minimize regret during the experiment
- Variants change over time
- You have many variants to test simultaneously

### CUPED (Controlled-experiment Using Pre-Experiment Data)
Variance reduction technique that uses pre-experiment data as a covariate to increase sensitivity by 20-50%, effectively reducing required sample size.

In [0]:
# ============================================================
# ADVANCED: SEQUENTIAL TESTING (O'Brien-Fleming Boundaries)
# ============================================================
# Use case: When you want to monitor results during an experiment
# without inflating false positive rate ("peeking problem")

def obrien_fleming_boundary(alpha, n_looks):
    """
    Calculate O'Brien-Fleming spending function boundaries.
    More conservative early, less conservative later.
    """
    boundaries = []
    for k in range(1, n_looks + 1):
        info_fraction = k / n_looks
        # O'Brien-Fleming approximation
        z_boundary = norm.ppf(1 - alpha/2) / np.sqrt(info_fraction)
        boundaries.append(z_boundary)
    return boundaries

# Simulate sequential monitoring
np.random.seed(42)
n_looks = 5  # Check 5 times during the experiment
total_n_per_group = 5000
alpha = 0.05

# O'Brien-Fleming boundaries
of_boundaries = obrien_fleming_boundary(alpha, n_looks)

# Simulate data accumulation
cumulative_z_stats = []
look_sizes = [total_n_per_group * (k/n_looks) for k in range(1, n_looks + 1)]

for i, n in enumerate(look_sizes):
    n_int = int(n)
    # Take first n observations from our data
    ctrl = ab_data[ab_data['group'] == 'control']['converted'].values[:n_int]
    trt = ab_data[ab_data['group'] == 'treatment']['converted'].values[:n_int]
    
    p_c, p_t = ctrl.mean(), trt.mean()
    p_pool = (ctrl.sum() + trt.sum()) / (2 * n_int)
    se = np.sqrt(p_pool * (1 - p_pool) * 2 / n_int)
    z = (p_t - p_c) / se if se > 0 else 0
    cumulative_z_stats.append(z)

print("=" * 60)
print("SEQUENTIAL TESTING: O'Brien-Fleming Boundaries")
print("=" * 60)
print(f"\n  Total planned sample: {total_n_per_group} per group")
print(f"  Number of interim looks: {n_looks}")
print(f"  Overall alpha: {alpha}")

print(f"\n  {'Look':<6}{'N/group':<10}{'Z-stat':<10}{'Boundary':<12}{'Decision'}")
print(f"  {'-'*50}")

early_stop = False
for i in range(n_looks):
    decision = 'STOP - Significant!' if abs(cumulative_z_stats[i]) > of_boundaries[i] else 'Continue'
    if abs(cumulative_z_stats[i]) > of_boundaries[i] and not early_stop:
        early_stop = True
        decision = 'STOP ✅ - Reject H₀!'
    print(f"  {i+1:<6}{int(look_sizes[i]):<10}{cumulative_z_stats[i]:<10.4f}{of_boundaries[i]:<12.4f}{decision}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
look_fractions = [k/n_looks for k in range(1, n_looks + 1)]

ax.plot(look_fractions, of_boundaries, 'r-o', linewidth=2, markersize=8, label='O\'Brien-Fleming Upper Boundary')
ax.plot(look_fractions, [-b for b in of_boundaries], 'r-o', linewidth=2, markersize=8, label='O\'Brien-Fleming Lower Boundary')
ax.plot(look_fractions, cumulative_z_stats, 'b-s', linewidth=2, markersize=10, label='Observed Z-statistic')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.fill_between(look_fractions, of_boundaries, 4, alpha=0.1, color='red', label='Rejection Region')
ax.fill_between(look_fractions, [-b for b in of_boundaries], -4, alpha=0.1, color='red')

ax.set_xlabel('Information Fraction (proportion of total sample)', fontsize=11)
ax.set_ylabel('Z-statistic', fontsize=11)
ax.set_title('Sequential Testing: O\'Brien-Fleming Monitoring\n(Boundaries protect against peeking inflation)', fontsize=12)
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(-4, 4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey Insight: O'Brien-Fleming boundaries are very strict early (hard to stop)")
print("but become close to the fixed-horizon boundary at the final look.")
print("This protects against inflated false positives from repeated peeking.")

In [0]:
# ============================================================
# ADVANCED: BAYESIAN A/B TESTING
# ============================================================
# Use case: When you want probability statements about which variant is better
# Advantages: No p-values, intuitive interpretation, handles early stopping naturally

from scipy.stats import beta as beta_dist

def bayesian_ab_test(control_conversions, control_total, 
                    treatment_conversions, treatment_total,
                    prior_alpha=1, prior_beta=1,  # Uniform prior
                    n_samples=100000):
    """
    Bayesian A/B test using Beta-Binomial conjugate model.
    
    Prior: Beta(alpha, beta) - default is uniform (uninformative)
    Posterior: Beta(alpha + successes, beta + failures)
    """
    # Posterior distributions
    alpha_c = prior_alpha + control_conversions
    beta_c = prior_beta + (control_total - control_conversions)
    
    alpha_t = prior_alpha + treatment_conversions
    beta_t = prior_beta + (treatment_total - treatment_conversions)
    
    # Sample from posteriors
    samples_control = np.random.beta(alpha_c, beta_c, n_samples)
    samples_treatment = np.random.beta(alpha_t, beta_t, n_samples)
    
    # P(Treatment > Control)
    prob_treatment_better = np.mean(samples_treatment > samples_control)
    
    # Expected lift
    lift_samples = (samples_treatment - samples_control) / samples_control
    expected_lift = np.mean(lift_samples)
    
    # Credible interval for the difference
    diff_samples = samples_treatment - samples_control
    ci_lower = np.percentile(diff_samples, 2.5)
    ci_upper = np.percentile(diff_samples, 97.5)
    
    # Expected loss (risk of choosing treatment if it's actually worse)
    loss_treatment = np.mean(np.maximum(samples_control - samples_treatment, 0))
    loss_control = np.mean(np.maximum(samples_treatment - samples_control, 0))
    
    return {
        'prob_treatment_better': prob_treatment_better,
        'expected_lift': expected_lift,
        'credible_interval': (ci_lower, ci_upper),
        'expected_loss_choosing_treatment': loss_treatment,
        'expected_loss_choosing_control': loss_control,
        'samples_control': samples_control,
        'samples_treatment': samples_treatment,
        'posterior_control': (alpha_c, beta_c),
        'posterior_treatment': (alpha_t, beta_t)
    }

# Run Bayesian test
control_conv = ab_data[ab_data['group'] == 'control']['converted'].sum()
control_n = len(ab_data[ab_data['group'] == 'control'])
treatment_conv = ab_data[ab_data['group'] == 'treatment']['converted'].sum()
treatment_n = len(ab_data[ab_data['group'] == 'treatment'])

bayes_result = bayesian_ab_test(control_conv, control_n, treatment_conv, treatment_n)

print("=" * 60)
print("BAYESIAN A/B TEST RESULTS")
print("Metric: Conversion Rate")
print("=" * 60)
print(f"\n  P(Treatment > Control):     {bayes_result['prob_treatment_better']:.4f} ({bayes_result['prob_treatment_better']*100:.1f}%)")
print(f"  Expected Relative Lift:     {bayes_result['expected_lift']*100:.2f}%")
print(f"  95% Credible Interval:      [{bayes_result['credible_interval'][0]*100:.3f}pp, {bayes_result['credible_interval'][1]*100:.3f}pp]")
print(f"\n  Expected Loss (choosing Treatment): {bayes_result['expected_loss_choosing_treatment']*100:.4f}pp")
print(f"  Expected Loss (choosing Control):   {bayes_result['expected_loss_choosing_control']*100:.4f}pp")
print(f"\n  Recommendation: {'Ship Treatment ✅' if bayes_result['prob_treatment_better'] > 0.95 else 'Continue Testing' if bayes_result['prob_treatment_better'] > 0.5 else 'Keep Control'}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot posterior distributions
x = np.linspace(0.09, 0.17, 1000)
alpha_c, beta_c = bayes_result['posterior_control']
alpha_t, beta_t = bayes_result['posterior_treatment']

axes[0].plot(x, beta_dist.pdf(x, alpha_c, beta_c), 'b-', linewidth=2, label=f'Control Posterior')
axes[0].plot(x, beta_dist.pdf(x, alpha_t, beta_t), 'r-', linewidth=2, label=f'Treatment Posterior')
axes[0].fill_between(x, beta_dist.pdf(x, alpha_c, beta_c), alpha=0.2, color='blue')
axes[0].fill_between(x, beta_dist.pdf(x, alpha_t, beta_t), alpha=0.2, color='red')
axes[0].set_xlabel('Conversion Rate', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Posterior Distributions of Conversion Rate\n(Bayesian Update)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot P(Treatment > Control) over time (simulate accumulation)
prob_over_time = []
sample_points = range(100, control_n + 1, 100)

for n in sample_points:
    c_conv = ab_data[ab_data['group'] == 'control']['converted'].values[:n].sum()
    t_conv = ab_data[ab_data['group'] == 'treatment']['converted'].values[:n].sum()
    
    # Quick Monte Carlo
    s_c = np.random.beta(1 + c_conv, 1 + n - c_conv, 5000)
    s_t = np.random.beta(1 + t_conv, 1 + n - t_conv, 5000)
    prob_over_time.append(np.mean(s_t > s_c))

axes[1].plot(list(sample_points), prob_over_time, 'g-', linewidth=2)
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% Decision Threshold')
axes[1].axhline(y=0.50, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Sample Size Per Group', fontsize=11)
axes[1].set_ylabel('P(Treatment > Control)', fontsize=11)
axes[1].set_title('Bayesian Belief Over Time\n(How confidence evolves with data)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("BAYESIAN vs FREQUENTIST COMPARISON")
print("="*60)
print(f"\n  {'Aspect':<30}{'Frequentist':<25}{'Bayesian'}")
print(f"  {'-'*75}")
print(f"  {'Result type':<30}{'p-value + CI':<25}{'Probability + Credible Int'}")
print(f"  {'Interpretation':<30}{'Reject/Fail to reject':<25}{'P(B>A) = X%'}")
print(f"  {'Early stopping':<30}{'Needs correction':<25}{'Natural (any time)'}")
print(f"  {'Prior knowledge':<30}{'Not used':<25}{'Incorporated'}")
print(f"  {'Sample size':<30}{'Fixed upfront':<25}{'Flexible'}")

In [0]:
# ============================================================
# ADVANCED: CUPED (Controlled-experiment Using Pre-Experiment Data)
# ============================================================
# Use case: Reduce variance and increase experiment sensitivity
# by leveraging pre-experiment behavior as a covariate

np.random.seed(42)

# Simulate pre-experiment data (prior 30 days behavior)
# Users who converted more before tend to convert more during experiment
pre_exp_conversions = np.random.binomial(30, 0.12, 10000) / 30  # Historical CVR per user

# During-experiment conversions (correlated with pre-experiment)
noise = np.random.normal(0, 0.08, 10000)
during_exp_raw = ab_data['converted'].values  # Our actual experiment data

# Add pre-experiment data to our dataset
ab_data['pre_exp_cvr'] = pre_exp_conversions

def cuped_adjustment(metric, covariate, group):
    """
    Apply CUPED variance reduction.
    
    Y_cuped = Y - theta * (X - E[X])
    where theta = Cov(Y, X) / Var(X)
    """
    # Calculate theta (optimal coefficient)
    theta = np.cov(metric, covariate)[0, 1] / np.var(covariate)
    
    # Adjusted metric
    y_cuped = metric - theta * (covariate - covariate.mean())
    
    return y_cuped, theta

# Apply CUPED to conversion data
control_mask = ab_data['group'] == 'control'
treatment_mask = ab_data['group'] == 'treatment'

metric = ab_data['converted'].values.astype(float)
covariate = ab_data['pre_exp_cvr'].values

y_cuped, theta = cuped_adjustment(metric, covariate, ab_data['group'])
ab_data['converted_cuped'] = y_cuped

# Compare variance before and after CUPED
var_before = metric.var()
var_after = y_cuped.var()
variance_reduction = (1 - var_after / var_before) * 100

print("=" * 60)
print("CUPED: VARIANCE REDUCTION TECHNIQUE")
print("=" * 60)
print(f"\n  Covariate: Pre-experiment conversion rate (30-day history)")
print(f"  Theta (optimal coefficient): {theta:.4f}")
print(f"\n  Variance BEFORE CUPED: {var_before:.6f}")
print(f"  Variance AFTER CUPED:  {var_after:.6f}")
print(f"  Variance Reduction:    {variance_reduction:.1f}%")

# Run t-test with and without CUPED
# Without CUPED
t_raw, p_raw = ttest_ind(
    ab_data[treatment_mask]['converted'],
    ab_data[control_mask]['converted']
)

# With CUPED
t_cuped, p_cuped = ttest_ind(
    ab_data[treatment_mask]['converted_cuped'],
    ab_data[control_mask]['converted_cuped']
)

print(f"\n  --- Comparison ---")
print(f"  {'Method':<20}{'T-stat':<12}{'P-value':<12}{'Significant'}")
print(f"  {'-'*55}")
print(f"  {'Raw (no CUPED)':<20}{t_raw:<12.4f}{p_raw:<12.6f}{'Yes ✅' if p_raw < 0.05 else 'No ❌'}")
print(f"  {'With CUPED':<20}{t_cuped:<12.4f}{p_cuped:<12.6f}{'Yes ✅' if p_cuped < 0.05 else 'No ❌'}")
print(f"\n  CUPED increases sensitivity by reducing noise from individual variation.")
print(f"  Effectively equivalent to {variance_reduction:.0f}% more data!")

In [0]:
# ============================================================
# MULTIPLE TESTING CORRECTION
# ============================================================
# Use case: When testing multiple metrics or variants simultaneously
# Problem: Testing 20 metrics at alpha=0.05 gives ~64% chance of at least one false positive!

from statsmodels.stats.multitest import multipletests

# Simulate testing multiple metrics
metrics_tested = {
    'Conversion Rate': result_ztest['p_value'],
    'Revenue Per User': result_ttest['p_value'],
    'Time on Page': result_time['p_value'],
    'Bounce Rate': two_proportion_ztest(ab_data, outcome_col='bounced')['p_value'],
}

# Add some additional "metrics" to demonstrate the problem
np.random.seed(99)
for i in range(6):
    # Simulate null metrics (no real effect)
    fake_p = np.random.uniform(0, 1)
    metrics_tested[f'Secondary Metric {i+1}'] = fake_p

metric_names = list(metrics_tested.keys())
p_values_raw = list(metrics_tested.values())

# Apply corrections
_, p_bonferroni, _, _ = multipletests(p_values_raw, method='bonferroni')
_, p_holm, _, _ = multipletests(p_values_raw, method='holm')
_, p_bh, _, _ = multipletests(p_values_raw, method='fdr_bh')  # Benjamini-Hochberg

print("=" * 80)
print("MULTIPLE TESTING CORRECTION")
print(f"Testing {len(metrics_tested)} metrics simultaneously")
print("=" * 80)

print(f"\n  Problem: With {len(metrics_tested)} tests at α=0.05:")
print(f"  P(at least 1 false positive) = 1 - (1-0.05)^{len(metrics_tested)} = {1 - (0.95**len(metrics_tested)):.3f}")

print(f"\n  {'Metric':<25}{'Raw p':<10}{'Bonferroni':<12}{'Holm':<10}{'BH (FDR)':<10}{'Raw Sig?':<10}{'BH Sig?'}")
print(f"  {'-'*85}")

for i, name in enumerate(metric_names):
    raw_sig = '✅' if p_values_raw[i] < 0.05 else '❌'
    bh_sig = '✅' if p_bh[i] < 0.05 else '❌'
    print(f"  {name:<25}{p_values_raw[i]:<10.4f}{p_bonferroni[i]:<12.4f}{p_holm[i]:<10.4f}{p_bh[i]:<10.4f}{raw_sig:<10}{bh_sig}")

print(f"\n--- Methods Explained ---")
print(f"  Bonferroni: Most conservative. Divides α by number of tests. Controls FWER.")
print(f"  Holm:       Stepwise improvement over Bonferroni. Less conservative, still controls FWER.")
print(f"  BH (FDR):   Controls False Discovery Rate. Best for exploratory analysis.")
print(f"\n  Recommendation: Use Bonferroni/Holm for primary metrics; BH for secondary/exploratory metrics.")

## 6. Industry Applications of A/B Testing

### Common Applications

| Industry | Use Case | Primary Metric | Test Type |
| --- | --- | --- | --- |
| **E-commerce** | Checkout flow redesign | Conversion rate, AOV | Z-test, Bootstrap |
| **SaaS** | Pricing page variants | Signup rate, MRR | Z-test, Revenue t-test |
| **Social Media** | Feed algorithm changes | Engagement, DAU/MAU | T-test, Ratio metrics |
| **Search** | Ranking algorithm | CTR, Session success rate | Interleaving, Z-test |
| **Streaming** | Recommendation engine | Watch time, Retention | T-test, Survival analysis |
| **Ride-sharing** | Surge pricing models | Rides completed, Revenue | Switchback experiments |
| **Email Marketing** | Subject line, send time | Open rate, CTR, Unsubscribe | Z-test |
| **Gaming** | Difficulty balancing | Retention D1/D7, Revenue | Mann-Whitney, Survival |

### Uncommon / Advanced Applications

| Scenario | Description | Challenge |
| --- | --- | --- |
| **Network effects** | Testing features where users interact (social, marketplace) | Interference between units |
| **Long-term effects** | Feature impact over months/years | Holdback groups, cohort analysis |
| **Supply-side experiments** | Testing on merchants/drivers (limited supply) | Small n, high heterogeneity |
| **Infrastructure changes** | Database migration, latency optimization | System-level metrics, no per-user randomization |
| **Pricing experiments** | Testing price points | Ethical concerns, revenue risk |
| **ML model rollout** | New ranking/recommendation model | Interleaving, credit assignment |
| **Physical world** | Store layout, packaging design | Geo-based randomization |
| **Healthcare / Clinical** | Drug trials, treatment protocols | Ethical constraints, adaptive designs |
| **Education** | Teaching methods, course content | Long feedback loops, ethical constraints |

In [0]:
# ============================================================
# APPLICATION: SEGMENTED ANALYSIS (HTE)
# ============================================================
# Key question: Does the treatment work DIFFERENTLY for different user segments?
# This is crucial for personalization and targeted rollout decisions

def segmented_analysis(data, segment_col, group_col='group', metric_col='converted'):
    """
    Perform A/B test within each segment to detect heterogeneous treatment effects.
    """
    segments = data[segment_col].unique()
    results = []
    
    for segment in sorted(segments):
        seg_data = data[data[segment_col] == segment]
        control = seg_data[seg_data[group_col] == 'control'][metric_col]
        treatment = seg_data[seg_data[group_col] == 'treatment'][metric_col]
        
        n_c, n_t = len(control), len(treatment)
        p_c, p_t = control.mean(), treatment.mean()
        
        # Z-test for this segment
        if n_c > 30 and n_t > 30:
            count = np.array([treatment.sum(), control.sum()])
            nobs = np.array([n_t, n_c])
            z_stat, p_val = proportions_ztest(count, nobs)
        else:
            z_stat, p_val = np.nan, np.nan
        
        results.append({
            'segment': segment,
            'n_control': n_c,
            'n_treatment': n_t,
            'cvr_control': p_c,
            'cvr_treatment': p_t,
            'absolute_lift': p_t - p_c,
            'relative_lift': (p_t - p_c) / p_c if p_c > 0 else np.nan,
            'p_value': p_val,
            'significant': p_val < 0.05 if not np.isnan(p_val) else False
        })
    
    return pd.DataFrame(results)

# Analyze by device type
device_results = segmented_analysis(ab_data, 'device')

print("=" * 70)
print("SEGMENTED ANALYSIS: Treatment Effect by Device")
print("=" * 70)
print()
display(device_results.style.format({
    'cvr_control': '{:.4f}',
    'cvr_treatment': '{:.4f}',
    'absolute_lift': '{:.4f}',
    'relative_lift': '{:.2%}',
    'p_value': '{:.4f}'
}).applymap(lambda x: 'background-color: lightgreen' if x == True else '', subset=['significant']))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conversion rate by segment
x_pos = np.arange(len(device_results))
width = 0.35

axes[0].bar(x_pos - width/2, device_results['cvr_control'], width, 
            label='Control', color='steelblue', alpha=0.8)
axes[0].bar(x_pos + width/2, device_results['cvr_treatment'], width, 
            label='Treatment', color='coral', alpha=0.8)
axes[0].set_xlabel('Device', fontsize=11)
axes[0].set_ylabel('Conversion Rate', fontsize=11)
axes[0].set_title('Conversion Rate by Device Segment', fontsize=12)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(device_results['segment'])
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Lift by segment with confidence
axes[1].barh(device_results['segment'], device_results['relative_lift'] * 100, 
             color=['green' if sig else 'gray' for sig in device_results['significant']],
             alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Relative Lift (%)', fontsize=11)
axes[1].set_title('Treatment Effect by Segment\n(Green = Significant, Gray = Not Significant)', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nKey Insight: Segmented analysis reveals WHERE the treatment works best.")
print("This enables targeted rollout — ship to segments with clear benefit first.")

In [0]:
# ============================================================
# APPLICATION: NOVELTY EFFECT DETECTION & TIME-BASED ANALYSIS
# ============================================================
# Key question: Is the treatment effect stable over time, or does it fade?
# Novelty effects are a major threat to experiment validity

def time_series_analysis(data, time_col='day', group_col='group', metric_col='converted'):
    """
    Analyze how the treatment effect evolves over experiment days.
    """
    days = sorted(data[time_col].unique())
    daily_results = []
    
    for day in days:
        day_data = data[data[time_col] == day]
        control = day_data[day_data[group_col] == 'control'][metric_col]
        treatment = day_data[day_data[group_col] == 'treatment'][metric_col]
        
        p_c, p_t = control.mean(), treatment.mean()
        lift = (p_t - p_c) / p_c if p_c > 0 else 0
        
        daily_results.append({
            'day': day,
            'cvr_control': p_c,
            'cvr_treatment': p_t,
            'absolute_lift': p_t - p_c,
            'relative_lift': lift,
            'n_users': len(day_data)
        })
    
    return pd.DataFrame(daily_results)

daily_analysis = time_series_analysis(ab_data)

# Check for novelty effect: is early lift different from late lift?
first_half = ab_data[ab_data['day'] <= 7]
second_half = ab_data[ab_data['day'] > 7]

lift_first = (first_half[first_half['group']=='treatment']['converted'].mean() - 
              first_half[first_half['group']=='control']['converted'].mean())
lift_second = (second_half[second_half['group']=='treatment']['converted'].mean() - 
               second_half[second_half['group']=='control']['converted'].mean())

print("=" * 60)
print("NOVELTY EFFECT ANALYSIS")
print("=" * 60)
print(f"\n  First Half (Days 1-7):  Absolute Lift = {lift_first:.4f} ({lift_first*100:.2f}pp)")
print(f"  Second Half (Days 8-14): Absolute Lift = {lift_second:.4f} ({lift_second*100:.2f}pp)")
print(f"  Difference: {(lift_first - lift_second)*100:.2f}pp")

if abs(lift_first - lift_second) > 0.01:
    print(f"  ⚠️  Potential novelty effect detected - lift differs between halves")
else:
    print(f"  ✅ Treatment effect appears stable over time")

# Cumulative conversion plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Daily lift over time
axes[0].plot(daily_analysis['day'], daily_analysis['relative_lift'] * 100, 'b-o', 
             linewidth=2, markersize=6)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].fill_between(daily_analysis['day'], 0, daily_analysis['relative_lift'] * 100, alpha=0.1)
axes[0].set_xlabel('Experiment Day', fontsize=11)
axes[0].set_ylabel('Relative Lift (%)', fontsize=11)
axes[0].set_title('Daily Treatment Effect Over Time\n(Watch for declining lift = novelty effect)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Cumulative metrics
cum_ctrl = []
cum_trt = []
for day in sorted(ab_data['day'].unique()):
    cum_data = ab_data[ab_data['day'] <= day]
    cum_ctrl.append(cum_data[cum_data['group']=='control']['converted'].mean())
    cum_trt.append(cum_data[cum_data['group']=='treatment']['converted'].mean())

days_sorted = sorted(ab_data['day'].unique())
axes[1].plot(days_sorted, np.array(cum_ctrl)*100, 'b-', linewidth=2, label='Control (Cumulative CVR)')
axes[1].plot(days_sorted, np.array(cum_trt)*100, 'r-', linewidth=2, label='Treatment (Cumulative CVR)')
axes[1].set_xlabel('Experiment Day', fontsize=11)
axes[1].set_ylabel('Cumulative Conversion Rate (%)', fontsize=11)
axes[1].set_title('Cumulative Conversion Rate\n(Lines should stabilize as sample grows)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Interpreting Results & Taking Action

### What Results Tell You

A/B test results yield several actionable pieces of information:

| Result Component | What It Tells You | Business Value |
| --- | --- | --- |
| **Point Estimate** (lift) | Best estimate of the treatment effect | Expected impact at full rollout |
| **Confidence Interval** | Range of plausible true effects | Risk assessment for decision |
| **P-value** | Strength of evidence against no-effect | Confidence in the direction |
| **Effect Size** | Practical magnitude of the difference | Is it worth the engineering/UX cost? |
| **Segment Effects** | Where it works best/worst | Targeted rollout strategy |
| **Guardrail Metrics** | Side effects and trade-offs | Risk of unintended consequences |

### Decision Framework

```
                    Significant?       Practically Meaningful?
                    ┌── YES ──────────── YES ───────────── → SHIP IT ✅
                    │                  └── NO ────────────── → Probably ship (low cost)
Experiment ────────┼
    Done            │                  ┌── Underpowered? ──── → Run longer / more traffic
                    └── NO ───────────┼
                                       └── Powered enough? ── → No effect - iterate on design
```

### Types of Actionable Outcomes

1. **Clear Winner** → Full rollout + document learnings
2. **Clear Loser** → Kill the variant + understand WHY
3. **Segment Winner** → Targeted rollout to winning segments
4. **Inconclusive** → Check power, consider extending or redesigning
5. **Guardrail Violation** → Do NOT ship regardless of primary metric win
6. **Surprising Result** → Replicate before acting

In [0]:
# ============================================================
# COMPREHENSIVE A/B TEST RESULTS DASHBOARD
# ============================================================
# This is what a real experiment report looks like

def generate_experiment_report(data):
    """
    Generate a comprehensive experiment report with all key metrics.
    """
    print("\n" + "═" * 70)
    print("║" + " A/B TEST EXPERIMENT REPORT ".center(68) + "║")
    print("═" * 70)
    
    # --- Experiment Overview ---
    print(f"\n{'EXPERIMENT OVERVIEW':=^70}")
    print(f"  Experiment:  Checkout Page Redesign")
    print(f"  Duration:    14 days")
    print(f"  Total Users: {len(data):,}")
    print(f"  Split:       50/50")
    print(f"  Status:      COMPLETED")
    
    # --- Sanity Checks (SRM) ---
    print(f"\n{'SANITY CHECKS':=^70}")
    n_c = len(data[data['group'] == 'control'])
    n_t = len(data[data['group'] == 'treatment'])
    expected_ratio = 0.5
    
    # Sample Ratio Mismatch test
    from scipy.stats import binom_test
    srm_p = stats.binom_test(n_c, n_c + n_t, expected_ratio)
    
    print(f"  Control:   {n_c:,} ({n_c/(n_c+n_t)*100:.1f}%)")
    print(f"  Treatment: {n_t:,} ({n_t/(n_c+n_t)*100:.1f}%)")
    print(f"  SRM Test p-value: {srm_p:.4f} {'(✅ No mismatch)' if srm_p > 0.01 else '(⚠️ SAMPLE RATIO MISMATCH!)'}")
    
    # --- Primary Metric ---
    print(f"\n{'PRIMARY METRIC: CONVERSION RATE':=^70}")
    result = two_proportion_ztest(data)
    print(f"  Control:        {result['control_rate']*100:.2f}%")
    print(f"  Treatment:      {result['treatment_rate']*100:.2f}%")
    print(f"  Absolute Lift:  {result['absolute_diff']*100:.2f}pp")
    print(f"  Relative Lift:  {result['relative_lift']*100:.2f}%")
    print(f"  P-value:        {result['p_value']:.6f}")
    print(f"  95% CI:         [{result['ci_95'][0]*100:.2f}pp, {result['ci_95'][1]*100:.2f}pp]")
    print(f"  Verdict:        {'SIGNIFICANT ✅' if result['significant'] else 'NOT SIGNIFICANT ❌'}")
    
    # --- Guardrail Metrics ---
    print(f"\n{'GUARDRAIL METRICS':=^70}")
    bounce_result = two_proportion_ztest(data, outcome_col='bounced')
    time_result = two_sample_ttest(data, metric_col='time_on_page')
    
    print(f"  {'Metric':<20}{'Control':<12}{'Treatment':<12}{'Lift':<12}{'Status'}")
    print(f"  {'-'*60}")
    print(f"  {'Bounce Rate':<20}{bounce_result['control_rate']*100:<12.2f}{bounce_result['treatment_rate']*100:<12.2f}{bounce_result['relative_lift']*100:<12.1f}%{'  ✅ Improved' if bounce_result['treatment_rate'] < bounce_result['control_rate'] else '  ⚠️ Degraded'}")
    print(f"  {'Time on Page (s)':<20}{time_result['control_mean']:<12.1f}{time_result['treatment_mean']:<12.1f}{time_result['relative_lift']*100:<12.1f}%{'  ✅ Increased' if time_result['treatment_mean'] > time_result['control_mean'] else '  ⚠️ Decreased'}")
    
    # --- Revenue Impact ---
    print(f"\n{'REVENUE IMPACT PROJECTION':=^70}")
    rev_result = two_sample_ttest(data, metric_col='revenue')
    rev_lift_per_user = rev_result['absolute_diff']
    
    # Project to annual impact
    daily_users = 10000  # Hypothetical
    annual_impact = rev_lift_per_user * daily_users * 365
    
    print(f"  Revenue/User Lift:    ${rev_lift_per_user:.4f}")
    print(f"  At {daily_users:,} users/day:")
    print(f"  Projected Annual Impact: ${annual_impact:,.0f}")
    print(f"  95% CI Annual Impact:    [${rev_result['ci_95'][0] * daily_users * 365:,.0f}, ${rev_result['ci_95'][1] * daily_users * 365:,.0f}]")
    
    # --- Recommendation ---
    print(f"\n{'RECOMMENDATION':=^70}")
    
    primary_sig = result['significant'] and result['relative_lift'] > 0
    guardrail_ok = bounce_result['treatment_rate'] <= bounce_result['control_rate'] * 1.05  # 5% tolerance
    
    if primary_sig and guardrail_ok:
        print(f"  ✅ SHIP THE TREATMENT")
        print(f"  Primary metric shows significant improvement with no guardrail degradation.")
        print(f"  Suggested rollout: Gradual (25% → 50% → 100%) over 1 week.")
    elif primary_sig and not guardrail_ok:
        print(f"  ⚠️ HOLD - Investigate guardrail degradation")
        print(f"  Primary metric improved but guardrails show concerning signals.")
    else:
        print(f"  ❌ DO NOT SHIP - Iterate on design")
        print(f"  No significant improvement detected. Consider redesigning the variant.")
    
    print(f"\n{'':=^70}")

# Generate the report
generate_experiment_report(ab_data)

In [0]:
# ============================================================
# PRACTICAL PITFALLS & VALIDATION CHECKS
# ============================================================
# These are the checks every data scientist should run BEFORE trusting results

print("=" * 70)
print("A/B TESTING PITFALLS & HOW TO DETECT THEM")
print("=" * 70)

# 1. AA TEST (pre-experiment validation)
print(f"\n{'1. AA TEST (Pre-Experiment Validation)':=^70}")
print("Run the same test on two 'control' groups to verify your system works.")
print("If AA test shows significance, your randomization or logging is broken!")

# Simulate AA test
np.random.seed(42)
aa_group1 = np.random.binomial(1, 0.12, 5000)
aa_group2 = np.random.binomial(1, 0.12, 5000)

count_aa = np.array([aa_group1.sum(), aa_group2.sum()])
nobs_aa = np.array([5000, 5000])
_, p_aa = proportions_ztest(count_aa, nobs_aa)

print(f"  AA Test p-value: {p_aa:.4f} {'(✅ System working correctly)' if p_aa > 0.05 else '(⚠️ SYSTEM ISSUE!)'}") 

# 2. SAMPLE RATIO MISMATCH (SRM)
print(f"\n{'2. SAMPLE RATIO MISMATCH (SRM)':=^70}")
print("If assignment is 50/50 but you observe 51/49, something is wrong.")
print("Common causes: Bot filtering, triggered experiments, logging bugs.")

n_c = len(ab_data[ab_data['group'] == 'control'])
n_t = len(ab_data[ab_data['group'] == 'treatment'])
expected = (n_c + n_t) / 2
chi2_srm = (n_c - expected)**2 / expected + (n_t - expected)**2 / expected
p_srm = 1 - stats.chi2.cdf(chi2_srm, df=1)

print(f"  Observed split: {n_c}/{n_t} (expected: {int(expected)}/{int(expected)})")
print(f"  SRM Chi-square: {chi2_srm:.4f}, p-value: {p_srm:.4f}")
print(f"  Status: {'(✅ No SRM detected)' if p_srm > 0.01 else '(⚠️ SRM DETECTED - DO NOT TRUST RESULTS!)'}")

# 3. PEEKING PROBLEM
print(f"\n{'3. PEEKING PROBLEM (Multiple Looks)':=^70}")
print("Checking results daily at α=0.05 over 14 days:")

# Simulate peeking under H0 (no real effect)
np.random.seed(55)
n_sims = 5000
false_positives_peek = 0

for _ in range(n_sims):
    # Generate full experiment data (H0 true)
    ctrl_full = np.random.binomial(1, 0.12, 5000)
    trt_full = np.random.binomial(1, 0.12, 5000)
    
    # Check every day (peek)
    found_sig = False
    for day in range(1, 15):
        n = int(5000 * day / 14)
        if n < 50:
            continue
        count_peek = np.array([trt_full[:n].sum(), ctrl_full[:n].sum()])
        nobs_peek = np.array([n, n])
        _, p_peek = proportions_ztest(count_peek, nobs_peek)
        if p_peek < 0.05:
            found_sig = True
            break
    
    if found_sig:
        false_positives_peek += 1

actual_fpr = false_positives_peek / n_sims
print(f"  Nominal α = 0.05")
print(f"  Actual False Positive Rate with daily peeking: {actual_fpr:.3f} ({actual_fpr*100:.1f}%)")
print(f"  Inflation factor: {actual_fpr / 0.05:.1f}x")
print(f"  ⚠️  Solution: Use sequential testing (O'Brien-Fleming) or Bayesian methods")

# 4. SIMPSON'S PARADOX
print(f"\n{'4. SIMPSONS PARADOX WARNING':=^70}")
print("Overall effect can REVERSE when you look at subgroups.")
print("Always check segment-level results alongside overall metrics.")
print("\nExample: Treatment might win overall because more mobile users")
print("entered treatment, and mobile users convert more — not because")
print("the treatment is actually better within any segment.")

# 5. PRACTICAL vs STATISTICAL SIGNIFICANCE
print(f"\n{'5. PRACTICAL vs STATISTICAL SIGNIFICANCE':=^70}")
print("A 0.01pp lift in CVR can be 'statistically significant' with n=1M")
print("but may not justify the engineering cost to maintain the feature.")
print(f"\n  Our result: +{result_ztest['absolute_diff']*100:.2f}pp lift")
print(f"  At 10K users/day: ~{result_ztest['absolute_diff'] * 10000:.0f} extra conversions/day")
print(f"  Is this worth the cost? → Depends on CLV and implementation cost")

## 8. Summary & Quick Reference

### Test Selection Flowchart

```
What type of metric?
│
├── Binary (converted/not, clicked/not)
│   ├── n > 30 per group → Z-test for proportions
│   └── n < 30 → Fisher's exact test
│
├── Continuous (revenue, time, score)
│   ├── Normal distribution (or large n) → Welch's t-test
│   ├── Skewed / non-normal → Mann-Whitney U or Bootstrap
│   └── Ratio metric (CTR = clicks/impressions) → Delta method
│
├── Categorical (multiple outcomes)
│   └── Chi-square test of independence
│
└── Time-to-event (days to churn, time to purchase)
    └── Log-rank test / Survival analysis
```

### Checklist Before Launching
- [ ] Hypothesis clearly defined
- [ ] Primary metric chosen (single OEC)
- [ ] Sample size calculated for desired MDE
- [ ] Guardrail metrics identified
- [ ] Randomization unit decided
- [ ] AA test passed
- [ ] Experiment duration planned (full weeks)
- [ ] Analysis plan pre-registered

### Checklist Before Calling Results
- [ ] Sample Ratio Mismatch (SRM) check passed
- [ ] Minimum sample size reached
- [ ] Full experiment duration elapsed
- [ ] Multiple testing correction applied (if needed)
- [ ] Novelty effects checked
- [ ] Segment analysis performed
- [ ] Guardrail metrics verified
- [ ] Practical significance assessed (not just statistical)

### Key Formulas

| Formula | Expression |
| --- | --- |
| Z-statistic (proportions) | $$Z = \frac{\hat{p}_T - \hat{p}_C}{\sqrt{\hat{p}(1-\hat{p})(\frac{1}{n_T} + \frac{1}{n_C})}}$$ |
| Confidence Interval | $$\hat{\delta} \pm Z_{\alpha/2} \cdot SE(\hat{\delta})$$ |
| Cohen's h (effect size) | $$h = 2\arcsin(\sqrt{p_T}) - 2\arcsin(\sqrt{p_C})$$ |
| Minimum Sample Size | $$n \geq \frac{(Z_{\alpha/2} + Z_{\beta})^2 \cdot 2\hat{p}(1-\hat{p})}{\delta^2}$$ |
| CUPED Adjustment | $$Y_{cuped} = Y - \theta(X - \bar{X})$$ where $$\theta = \frac{Cov(Y,X)}{Var(X)}$$ |

### Resources
- *Trustworthy Online Controlled Experiments* by Ron Kohavi, Diane Tang, Ya Xu
- *Statistical Methods in Online A/B Testing* by Georgi Georgiev
- Google's "Overlapping Experiment Infrastructure" paper
- Microsoft's ExP platform documentation

In [0]:
# ============================================================
# END-TO-END REUSABLE A/B TEST ANALYSIS CLASS
# ============================================================
# A production-ready class you can use for any A/B test

class ABTestAnalyzer:
    """
    Complete A/B test analysis toolkit.
    
    Usage:
        analyzer = ABTestAnalyzer(data, group_col='group', 
                                  control_label='control', treatment_label='treatment')
        analyzer.run_test(metric_col='converted', metric_type='proportion')
        analyzer.summary()
    """
    
    def __init__(self, data, group_col='group', 
                 control_label='control', treatment_label='treatment'):
        self.data = data
        self.group_col = group_col
        self.control_label = control_label
        self.treatment_label = treatment_label
        self.control = data[data[group_col] == control_label]
        self.treatment = data[data[group_col] == treatment_label]
        self.results = {}
    
    def sanity_check(self):
        """Run pre-analysis sanity checks."""
        n_c, n_t = len(self.control), len(self.treatment)
        total = n_c + n_t
        
        # SRM check
        expected = total / 2
        chi2_srm = (n_c - expected)**2 / expected + (n_t - expected)**2 / expected
        p_srm = 1 - stats.chi2.cdf(chi2_srm, df=1)
        
        self.results['sanity'] = {
            'n_control': n_c,
            'n_treatment': n_t,
            'ratio': n_c / n_t,
            'srm_p_value': p_srm,
            'srm_passed': p_srm > 0.01
        }
        return self.results['sanity']
    
    def run_test(self, metric_col, metric_type='proportion', alpha=0.05):
        """
        Run the appropriate statistical test.
        
        metric_type: 'proportion' for binary, 'continuous' for numeric
        """
        control_metric = self.control[metric_col]
        treatment_metric = self.treatment[metric_col]
        
        if metric_type == 'proportion':
            count = np.array([treatment_metric.sum(), control_metric.sum()])
            nobs = np.array([len(treatment_metric), len(control_metric)])
            z_stat, p_value = proportions_ztest(count, nobs)
            
            p_c, p_t = control_metric.mean(), treatment_metric.mean()
            se = np.sqrt(p_c*(1-p_c)/len(control_metric) + p_t*(1-p_t)/len(treatment_metric))
            ci = ((p_t - p_c) - 1.96*se, (p_t - p_c) + 1.96*se)
            
            self.results['test'] = {
                'metric': metric_col,
                'type': 'Two-Proportion Z-Test',
                'control_value': p_c,
                'treatment_value': p_t,
                'absolute_diff': p_t - p_c,
                'relative_lift': (p_t - p_c) / p_c,
                'statistic': z_stat,
                'p_value': p_value,
                'ci_95': ci,
                'significant': p_value < alpha
            }
        
        elif metric_type == 'continuous':
            t_stat, p_value = ttest_ind(treatment_metric, control_metric, equal_var=False)
            
            mean_c, mean_t = control_metric.mean(), treatment_metric.mean()
            se = np.sqrt(control_metric.var()/len(control_metric) + 
                        treatment_metric.var()/len(treatment_metric))
            ci = ((mean_t - mean_c) - 1.96*se, (mean_t - mean_c) + 1.96*se)
            
            self.results['test'] = {
                'metric': metric_col,
                'type': "Welch's T-Test",
                'control_value': mean_c,
                'treatment_value': mean_t,
                'absolute_diff': mean_t - mean_c,
                'relative_lift': (mean_t - mean_c) / mean_c if mean_c != 0 else np.nan,
                'statistic': t_stat,
                'p_value': p_value,
                'ci_95': ci,
                'significant': p_value < alpha
            }
        
        return self.results['test']
    
    def summary(self):
        """Print a clean summary of results."""
        if 'sanity' not in self.results:
            self.sanity_check()
        
        s = self.results.get('sanity', {})
        t = self.results.get('test', {})
        
        print("\n" + "═" * 60)
        print(f" A/B TEST RESULT: {t.get('metric', 'N/A')} ".center(60))
        print("═" * 60)
        print(f"  Test Type:       {t.get('type', 'N/A')}")
        print(f"  Sample:          {s.get('n_control', 0):,} control / {s.get('n_treatment', 0):,} treatment")
        print(f"  SRM Check:       {'PASS ✅' if s.get('srm_passed') else 'FAIL ⚠️'}")
        print(f"  Control:         {t.get('control_value', 0):.4f}")
        print(f"  Treatment:       {t.get('treatment_value', 0):.4f}")
        print(f"  Lift:            {t.get('relative_lift', 0)*100:.2f}% relative")
        print(f"  P-value:         {t.get('p_value', 1):.6f}")
        if t.get('ci_95'):
            print(f"  95% CI:          [{t['ci_95'][0]:.6f}, {t['ci_95'][1]:.6f}]")
        print(f"  Significant:     {'YES ✅' if t.get('significant') else 'NO ❌'}")
        print("═" * 60)


# --- DEMO: Using the class ---
print("=" * 60)
print("DEMO: Using ABTestAnalyzer Class")
print("=" * 60)

# Test 1: Conversion rate
analyzer = ABTestAnalyzer(ab_data)
analyzer.sanity_check()
analyzer.run_test('converted', metric_type='proportion')
analyzer.summary()

# Test 2: Revenue
analyzer2 = ABTestAnalyzer(ab_data)
analyzer2.run_test('revenue', metric_type='continuous')
analyzer2.summary()

# Test 3: Time on page
analyzer3 = ABTestAnalyzer(ab_data)
analyzer3.run_test('time_on_page', metric_type='continuous')
analyzer3.summary()

print("\n\n✅ This ABTestAnalyzer class is ready for production use!")
print("   Copy it into your codebase and call .run_test() on any metric.")